# slice-view-mutation — ex2: zero a row in-place and prove storage aliasing via data_ptr

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `slice-view-mutation`. Running the final beacon cell reports progress against the `PyTorch: Slice view mutation` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Slice view mutation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`slice-view-mutation`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "slice-view-mutation"
DD_SUBTOPIC = "PyTorch: Slice view mutation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Slice view mutation — quick refresher

Slicing returns a **view** sharing storage with the source. Writes through the view alias the source — and crucially, `data_ptr()` lets you *prove* the aliasing from the outside:

```python
row = x[2]
assert row.data_ptr() == x.data_ptr() + 2 * x.stride(0) * x.element_size()
```

The previous drill (ex1) zeroed the **diagonal** via `mat.diagonal()[:] = 0`. This drill zeroes a **row** via the more direct `mat[i, :] = val` syntax and uses `data_ptr()` to assert the source storage was mutated, not replaced.

### Exercise 2 — zero a row in-place and prove storage aliasing via data_ptr

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Analyze
> LO: Analyze the storage-aliasing property of slice-views by mutating a single row of a matrix in place via `mat[i, :] = val` and verifying `data_ptr()` is unchanged across the write.
> Keywords: slice, view, in-place, row, data_ptr
> ```

**KCs targeted:** `slice-returns-view`, `view-writes-alias-source`

Implement `ex2_zero_row_inplace(mat, i, value)`.

Given a `(R, C)` float tensor `mat`, a row index `i`, and a scalar `value`, set every entry of row `i` to `value` **in place** using the `mat[i, :] = value` slice-assignment syntax. Then return `(mat, did_alias)` where `did_alias` is a bool: `True` iff `mat.data_ptr()` was unchanged across the write (proving the write went to the original storage, not a fresh allocation).

**Rules.**
1. No reassignment of `mat` — work in place.
2. Capture `mat.data_ptr()` BEFORE the write, then again AFTER, and set `did_alias = (ptr_before == ptr_after)`.
3. Other rows must be untouched.

Inputs:
- `mat`: `(R, C)` float tensor.
- `i`: int row index.
- `value`: Python float (gets broadcast across the row).

Output: `(mat, did_alias)` — the same tensor object plus the aliasing flag.

In [ ]:
def ex2_zero_row_inplace(mat: Tensor, i: int, value: float):
    ptr_before = mat.data_ptr()
    mat[i, :] = value
    ptr_after = mat.data_ptr()
    did_alias = (ptr_before == ptr_after)
    return mat, did_alias


<details><summary>Solution</summary>

```python
def ex2_zero_row_inplace(mat: Tensor, i: int, value: float):
    ptr_before = mat.data_ptr()
    mat[i, :] = value
    ptr_after = mat.data_ptr()
    did_alias = (ptr_before == ptr_after)
    return mat, did_alias
```

**Why `data_ptr()` is the cleanest aliasing proof.** Two tensors share storage iff their `data_ptr()` values are equal (for the same offset). Capturing it before and after a write proves the mutation went to the original allocation — it's how PyTorch's own tests verify in-place ops.

**Difference from ex1.** ex1 used `mat.diagonal()[:] = 0` — an indexed view of a non-contiguous axis. ex2 uses the more common `mat[i, :] = val` row-slice form and adds the explicit `data_ptr()` check, which catches the subtle bug where someone writes `mat = mat.clone(); mat[i] = val` and breaks aliasing without noticing.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()